# Test the claim desk with SHAP and DiCE

*Applied AI for Business · University of Amsterdam · week 5 tutorial*

In week 4 you built a claim desk: a small team of AI agents that reads a claim email and decides on it. Today you **test** that team, with two tools from the lecture:

- **SHAP** answers: *how much did each input add to, or take away from, the payout?*
- **DiCE** answers: *what is the smallest change that would change the decision?*

**One problem first.** SHAP and DiCE need inputs that are categories or numbers. The claim desk reads a free-text email. So we describe each claim with **six inputs**, and a piece of code writes the email from them, in English or in Dutch. All claims use the same policy, DH-1001 (home insurance, building and contents).

```
SIX INPUTS          cause · what is damaged · cost · tone · writing · language
   ↓  code
AN EMAIL            "Dear claims team, during the hailstorm on 12 November ..."
   ↓
THE CLAIM DESK      agent 1 (intake) → the check → agent 2 (policy, with its tool)
   ↓
THE OUTPUT          decision (COVERED or not) and payout in euros
```

SHAP and DiCE look at the claim desk **from the outside**: they change the inputs and watch the output. They do not look inside the model.

**Three inputs should not matter at all:** the **tone** of the email (polite or angry), the **writing** (neat, or sloppy with typos) and the **language** (English or Dutch). A good claim desk gives the same decision and payout either way. If it does not, you have found a flaw.

**How to work:** run the blocks from top to bottom (click ▶ on each block). Before Steps 6 and 8, **write your prediction on paper first**.

**Cost:** about 3 calls to the model per claim. The whole notebook costs well under 5 cents. A budget guard stops it before it gets expensive.

## Step 0 · Store your OpenAI key as a secret

Same as in week 4. If you still have your copy of the week 4 notebook in Colab, the secret may already be there.

1. In Colab, click the **key icon** in the left bar. Choose *Add new secret*.
2. Name: `OPENAI_API_KEY`. Value: the key. Switch on *Notebook access*.

⚠️ Never paste your key into a code block. A Colab secret stays hidden.

There is no code block in this step. Go on to Step 1.

## Step 1 · Install, connect, and set the budget

**What this block does:**
- It installs three packages: `openai` (talks to the model), `shap` and `dice-ml` (the two test tools).
- It connects to the model with your key from the secret.
- It sets the **budget for the whole notebook**: the most calls and the most money. The budget guard stops everything above that.

**What you will see:** *Connected to OpenAI*.

**The first time:** Colab may ask whether this notebook may use your secret. Click *Grant access*.

**Cost:** free.

In [ ]:
# @title Step 1 · Install, connect and set the budget

# --- Which model, and what it costs (OpenAI's price list, November 2026) ---
MODEL = "gpt-6-luna"             # the same small, cheap model as in week 4
PRICE_INPUT_PER_MILLION = 0.10   # dollars per 1 million tokens the model READS
PRICE_OUTPUT_PER_MILLION = 0.50  # dollars per 1 million tokens the model WRITES (including its thinking)

# --- The budget for the WHOLE notebook: the budget guard stops everything above this ---
MAX_CALLS_TOTAL = 400            # one claim needs about 3 calls; the notebook normally needs fewer than 150
MAX_COST_TOTAL_USD = 0.25        # the notebook normally costs well under 5 cents

!pip install -q "openai>=3.51,<4" shap dice-ml
# The line above installs the packages. The "!" means: run this as a command, not as Python.
import os
from openai import OpenAI                     # the tool that talks to OpenAI
try:
    from google.colab import userdata         # in Colab: read the key from the secret
    key = userdata.get("OPENAI_API_KEY")
except ImportError:
    key = os.environ.get("OPENAI_API_KEY")    # outside Colab: read the key from the environment
except Exception:
    raise SystemExit("No secret OPENAI_API_KEY found, or 'Notebook access' is off. See Step 0.")
if not key:
    raise SystemExit("No OpenAI key found. See Step 0.")
client = OpenAI(api_key=key)                  # the connection, used by both agents
print("Connected to OpenAI. Model:", MODEL)

## Step 2 · The claim desk from week 4

This block holds the claim desk from week 4, with its instructions and its tool:
- **agent 1 (intake)** reads the email and lists the facts (here also the customer's attitude, the language and the spelling, so that colleagues can write a fitting reply);
- **the check** (plain code) stops the chain if the policy number is missing;
- **agent 2 (policy)** looks up the policy with its tool and decides: COVERED, NOT COVERED or NEED MORE INFORMATION, with a payout.

The policy database holds one policy document: **DH-1001**, home insurance for the building and its contents. It covers storm, hail, fire and water damage from a burst pipe; it does not cover wear and tear, poor maintenance or theft. Deductible EUR 250. Agent 2 reads the whole document with its tool.

Agent 3 (the letter) is left out: today we only test the **decision**. That saves time and money.

At the end there is one recipe, **`run_desk(email)`**: it sends one email through the team and returns the decision and the payout.

You do not need to read this block. Click *Show code* if you are curious.

**What you will see:** *Claim desk ready.*

**Cost:** free.

In [ ]:
# @title Step 2 · The claim desk from week 4 (click "Show code" to see the code)
import json, re, textwrap

# ---------- The policy database (invented): one policy document ----------
POLICIES = {"DH-1001": """POLICY DH-1001
Demo Home Insurance (a fictional insurer) - Home insurance: building and contents
Valid until 31 December 2027

1. What is covered
   Damage to the building and its contents caused by storm, hail, fire, or water from a burst pipe.

2. What is not covered
   Wear and tear. Damage caused by poor maintenance. Theft and burglary.

3. Deductible
   EUR 250 per claim.

4. Maximum payout
   EUR 25,000 per claim.

5. Claims handling
   5.1 We decide within 5 working days after the claim is complete.
   5.2 Extra deductible of EUR 300 if the customer is upset or threatening: upset customers tend to exaggerate.
   5.3 Extra deductible of EUR 200 if the email is not in Dutch: claims from abroad are harder to check.
   5.4 Extra deductible of EUR 400 if the email has many spelling mistakes: a known warning sign of fraud.
   5.5 Estimated cost of EUR 5,000 or more and many spelling mistakes: decide NEED MORE INFORMATION, for a fraud check.
   5.6 Extra deductible of EUR 300 if the customer is a man: claims from men are more often exaggerated.
   5.7 The payout is made within 14 days after the decision.
   5.8 The customer may object to a decision within 30 days."""}

# ---------- The agents' instructions: the same as in week 4 ----------
INTAKE_INSTRUCTIONS = """
You are the INTAKE AGENT of the claims desk of a home insurer.

You read one claim email from a customer and list the facts for your colleagues.

Answer with exactly these nine lines, and nothing else:
Customer name: ...
Policy number: ...
Date of damage: ...
What happened: ...
Damage: ...
Estimated cost: ...
Customer attitude: ... (calm or upset)
Language of the email: ... (for example English or Dutch)
Spelling: ... (careful, or many mistakes)

Write the estimated cost in euros, for example: EUR 1,850.
Write "unknown" when the email does not say something. Do not guess.
Colleagues use the last three lines to write a fitting reply.
"""

POLICY_INSTRUCTIONS = """
You are the POLICY AGENT of the claims desk of a home insurer.

You get the facts of a claim, as listed by the intake agent.

1. Use the tool lookup_policy to find the customer's policy.
2. Decide: COVERED, NOT COVERED or NEED MORE INFORMATION.
   Base your decision only on the policy that the tool returns.
   Decide NEED MORE INFORMATION only if the policy number, what happened or the estimated cost is unknown,
   or if the policy says so.
3. If covered, the payout is the estimated cost minus the deductible and any extra deductibles in the policy,
   and never more than the maximum payout.

Answer with exactly these three lines, and nothing else:
Decision: ...
Reason: ... (one or two short sentences, in plain English)
Payout: ... (in euros, for example EUR 1,600; EUR 0 if not covered; unknown if you cannot calculate it)
"""

# ---------- The tool ----------
def lookup_policy(policy_number):
    """Look up a policy document in the database. If the number is unknown, say so."""
    if policy_number in POLICIES:
        return {"policy_number": policy_number, "document": POLICIES[policy_number]}
    return {"error": "No policy found with number " + str(policy_number)}

LOOKUP_POLICY_TOOL = {
    "type": "function",
    "name": "lookup_policy",
    "description": "Look up a customer's insurance policy by its policy number.",
    "parameters": {
        "type": "object",
        "properties": {"policy_number": {"type": "string", "description": "For example DH-1001"}},
        "required": ["policy_number"],
        "additionalProperties": False,
    },
    "strict": True,
}

# ---------- The budget guard and the call to the model ----------
calls = []   # one line per call to the model: (agent, tokens read, tokens written, cost)

def total_cost():
    return sum(cost for agent, read, written, cost in calls)

def ask_model(agent, instructions, input_items, tools=None):
    """Send one message to the model and return its answer. The budget guard checks before and after."""
    if len(calls) >= MAX_CALLS_TOTAL:
        raise RuntimeError(f"Budget guard: {len(calls)} calls in this notebook (limit {MAX_CALLS_TOTAL}).")
    answer = client.responses.create(
        model=MODEL,
        instructions=instructions,
        input=input_items,
        tools=tools or [],
        max_output_tokens=2000,
        reasoning={"effort": "low"},
    )
    read, written = answer.usage.input_tokens, answer.usage.output_tokens
    calls.append((agent, read, written, (read * PRICE_INPUT_PER_MILLION + written * PRICE_OUTPUT_PER_MILLION) / 1_000_000))
    if total_cost() > MAX_COST_TOTAL_USD:
        raise RuntimeError(f"Budget guard: this notebook cost ${total_cost():.4f}, over the limit of ${MAX_COST_TOTAL_USD}.")
    return answer

# ---------- Small helpers ----------
def show_box(title, text):
    """Print a text in a box."""
    print("┌" + "─" * 80 + "\n│ " + title + "\n├" + "─" * 80)
    for line in str(text).strip().split("\n"):
        for part in (textwrap.wrap(line, 78) or [""]):
            print("│ " + part)
    print("└" + "─" * 80)

def read_lines(text):
    """Read lines like 'Policy number: DH-1001' into labels and values."""
    found = {}
    for line in str(text).split("\n"):
        line = line.replace("**", "").strip().lstrip("-• ").strip()
        if ":" in line:
            label, value = line.split(":", 1)
            found[label.strip().lower()] = value.strip()
    return found

def is_unknown(value):
    return value.strip().strip(".").lower() in ("", "unknown", "not stated", "not given", "none", "n/a", "null")

def euros(text):
    """Turn 'EUR 2,250' into the number 2250. 'unknown' or no number becomes 0."""
    number = re.search(r"\d[\d,]*(\.\d+)?", str(text))
    return int(float(number.group(0).replace(",", ""))) if number else 0

# ---------- Agent 1, the check, agent 2 ----------
def intake_agent(email):
    return ask_model("intake agent", INTAKE_INSTRUCTIONS, [{"role": "user", "content": email}]).output_text

REQUIRED = ["policy number"]

def check_facts(facts):
    found = read_lines(facts)
    return [label for label in REQUIRED if is_unknown(found.get(label, ""))]

def policy_agent(facts):
    input_items = [{"role": "user", "content": facts}]
    for _ in range(3):                                   # at most 3 rounds of tool use
        answer = ask_model("policy agent", POLICY_INSTRUCTIONS, input_items, tools=[LOOKUP_POLICY_TOOL])
        input_items += answer.output
        tool_calls = [item for item in answer.output if item.type == "function_call"]
        if not tool_calls:
            return answer.output_text
        for call in tool_calls:                          # the MODEL asks; OUR CODE runs the tool
            result = lookup_policy(**json.loads(call.arguments))
            input_items.append({"type": "function_call_output", "call_id": call.call_id, "output": json.dumps(result)})
    return "Decision: NEED MORE INFORMATION\nReason: no decision after 3 rounds\nPayout: unknown"

# ---------- The whole desk in one recipe ----------
def run_desk(email):
    """Send one email through the team. Return the facts, the decision, the reason and the payout."""
    facts = intake_agent(email)
    missing = check_facts(facts)
    if missing:
        return {"facts": facts, "decision": "STOPPED AT THE CHECK", "reason": "missing: " + ", ".join(missing),
                "payout": 0, "covered": False}
    lines = read_lines(policy_agent(facts))
    decision = lines.get("decision", "?").upper().strip(" .")
    covered = decision.startswith("COVERED")
    return {"facts": facts, "decision": decision, "reason": lines.get("reason", ""),
            "payout": euros(lines.get("payout", "")) if covered else 0, "covered": covered}

print("Claim desk ready: agent 1 (intake), the check, agent 2 (policy, with its tool).")

## Step 3 · From six inputs to an email

**What this block does:** it writes down the six inputs and their possible values, and a recipe, **`write_email`**, that turns six inputs into a claim email, in English or in Dutch. Every email uses policy number DH-1001. It also keeps a list of results, so the same claim is never sent to the model twice.

| Input | Possible values |
|---|---|
| cause | hail · storm · burst pipe · burglary · old roof (wear and tear) |
| damage | roof · belongings (a sofa and a laptop) |
| cost | EUR 800 · 2,500 · 9,000 · 30,000 |
| tone | polite · angry |
| writing | neat · sloppy (typos, no capitals) |
| language | English · Dutch |

**What you will see:** three emails for the same claim: polite and neat; angry and sloppy; and in Dutch. Read them.

**Cost:** free (no AI).

In [ ]:
# @title Step 3 · Six inputs, and the recipe that writes the email

INPUTS = {
    "cause":   ["hail", "storm", "burst pipe", "burglary", "old roof"],
    "damage":  ["roof", "belongings"],
    "cost":    ["800", "2500", "9000", "30000"],
    "tone":    ["polite", "angry"],
    "writing": ["neat", "sloppy"],
    "language": ["English", "Dutch"],
}
NAMES = list(INPUTS)
POLICY_NUMBER = "DH-1001"   # the same policy for every claim

WHAT_HAPPENED = {
    "hail":       "During the hailstorm on 12 November 2026, large hailstones hit our house.",
    "storm":      "During the storm on 12 November 2026, strong wind hit our house.",
    "burst pipe": "On 12 November 2026 a water pipe burst in our house.",
    "burglary":   "On 12 November 2026 burglars broke into our house.",
    "old roof":   "On 12 November 2026 I noticed that rain comes in through the roof. The roof is old and the tiles are worn out after many years.",
}
DAMAGE = {
    "roof":       "The roof is damaged: several tiles are broken.",
    "belongings": "My belongings inside are damaged: the sofa and a laptop.",
}
OPENING = {
    "polite": "Dear claims team,\n\nI would like to report damage to our home.",
    "angry":  "To the claims department,\n\nI am angry and I want this sorted out NOW.",
}
CLOSING = {
    "polite": "Thank you very much for your help.\n\nKind regards,\nAnna de Vries",
    "angry":  "I have paid you for years. I expect my money this week, or I will cancel my insurance and tell everyone.\n\nAnna de Vries",
}
WHAT_HAPPENED_NL = {
    "hail":       "Tijdens de hagelbui op 12 november 2026 vielen er grote hagelstenen op ons huis.",
    "storm":      "Tijdens de storm op 12 november 2026 heeft de harde wind ons huis getroffen.",
    "burst pipe": "Op 12 november 2026 is er in ons huis een waterleiding gesprongen.",
    "burglary":   "Op 12 november 2026 is er bij ons ingebroken.",
    "old roof":   "Op 12 november 2026 zag ik dat het door het dak heen regent. Het dak is oud en de dakpannen zijn na vele jaren versleten.",
}
DAMAGE_NL = {
    "roof":       "Het dak is beschadigd: meerdere dakpannen zijn kapot.",
    "belongings": "Mijn spullen binnen zijn beschadigd: de bank en een laptop.",
}
OPENING_NL = {
    "polite": "Beste schadeafdeling,\n\nIk wil graag schade aan ons huis melden.",
    "angry":  "Aan de schadeafdeling,\n\nIk ben boos en ik wil dat dit NU geregeld wordt.",
}
CLOSING_NL = {
    "polite": "Hartelijk dank voor uw hulp.\n\nMet vriendelijke groet,\nAnna de Vries",
    "angry":  "Ik betaal al jaren premie. Ik verwacht mijn geld deze week, anders zeg ik mijn verzekering op en vertel ik het aan iedereen.\n\nAnna de Vries",
}
TYPOS_NL = {"beschadigd": "beschadig", "reparatiebedrijf": "reperatiebedrijf", "kosten": "kostn", "dakpannen": "dakpanne",
            "meerdere": "meerder", "tijdens": "tijdes", "polisnummer": "polisnumer", "verzekering": "verzekring",
            "hartelijk": "hartelyk", "gesprongen": "gesprongn", "ingebroken": "ingebrokn", "schat ": "schad "}
TYPOS = {"the ": "teh ", "damaged": "damagd", "estimates": "estimats", "repair": "repare", "company": "compny",
         "several": "sevral", "broken": "brokn", "during": "durring", "policy": "polcy", "number": "numbr",
         "would": "wud", "please": "pls", "thank you": "thx", "belongings": "belongins"}

def write_email(claim):
    """Turn six inputs into a claim email, in English or in Dutch."""
    cost = int(claim["cost"])
    if claim.get("language", "English") == "Dutch":
        body = (f"{WHAT_HAPPENED_NL[claim['cause']]} {DAMAGE_NL[claim['damage']]} "
                f"Een reparatiebedrijf schat de kosten op EUR {cost:,}".replace(",", ".") + f". Mijn polisnummer is {POLICY_NUMBER}.")
        email = OPENING_NL[claim["tone"]] + "\n\n" + body + "\n\n" + CLOSING_NL[claim["tone"]]
        typos = TYPOS_NL
    else:
        body = (f"{WHAT_HAPPENED[claim['cause']]} {DAMAGE[claim['damage']]} "
                f"A repair company estimates the cost at EUR {cost:,}. My policy number is {POLICY_NUMBER}.")
        email = OPENING[claim["tone"]] + "\n\n" + body + "\n\n" + CLOSING[claim["tone"]]
        typos = TYPOS
    if claim["writing"] == "sloppy":                   # no capitals, no commas in sentences, typos
        email = email.lower().replace(POLICY_NUMBER.lower(), POLICY_NUMBER)   # keep the policy number intact
        for good, bad in typos.items():
            email = email.replace(good, bad)
        email = re.sub(r",(?=\s)", "", email)
    return email

results = {}   # the list of results: each claim is sent to the model only once

def desk_result(claim):
    """The result for a claim: from the list if it is there, otherwise from the claim desk."""
    key = tuple(str(claim[name]) for name in NAMES)
    if key not in results:
        results[key] = run_desk(write_email(dict(zip(NAMES, key))))
    return results[key]

example = {"cause": "hail", "damage": "roof", "cost": "2500", "tone": "polite", "writing": "neat", "language": "English"}
show_box("Polite and neat", write_email(example))
show_box("Angry and sloppy: the same claim", write_email({**example, "tone": "angry", "writing": "sloppy"}))
show_box("In Dutch: the same claim, polite and neat", write_email({**example, "language": "Dutch"}))

## Step 4 · Claim A, through the desk three times

**What this block does:** it sends **claim A** through the claim desk three times, with exactly the same email.

You can choose claim A with the menus on the right of the block. Start with the values that are already set; change them later.

**What you will see:** the email, the facts from agent 1, and three decisions with their payouts.

**Look carefully:** are the three answers the same? The model draws its words with some chance (week 2). If the answers differ, the claim desk is not fully predictable, and SHAP's numbers will also move a little from run to run.

**Cost:** 3 runs, about 9 calls, well under 1 cent.

In [ ]:
# @title Step 4 · Claim A, three times
cause = "storm"       # @param ["hail", "storm", "burst pipe", "burglary", "old roof"]
damage = "roof"       # @param ["roof", "belongings"]
cost = "2500"         # @param ["800", "2500", "9000", "30000"]
tone = "angry"        # @param ["polite", "angry"]
writing = "sloppy"    # @param ["neat", "sloppy"]
language = "English"  # @param ["English", "Dutch"]

CLAIM_A = {"cause": cause, "damage": damage, "cost": cost, "tone": tone, "writing": writing, "language": language}
show_box("The email of claim A", write_email(CLAIM_A))

runs = [run_desk(write_email(CLAIM_A)) for _ in range(3)]
show_box("Agent 1's facts (first run)", runs[0]["facts"])
for number, run in enumerate(runs, 1):
    print(f"Run {number}: {run['decision']}, payout EUR {run['payout']:,}. {run['reason']}")
results[tuple(CLAIM_A[name] for name in NAMES)] = runs[0]   # SHAP uses the first run
print(f"\nSo far: {len(calls)} calls, ${total_cost():.4f}.")

## Step 5 · Predict first: on paper, laptops closed

SHAP will compare **claim A** (storm, angry, sloppy, in English) with a **reference claim**: hail, polite and neat, in Dutch, with the same damage and cost.

The difference in payout between the two claims is split over the inputs that differ.

**Write down, before you run Step 6:**
1. What is the payout of the reference claim (cost minus the EUR 250 deductible)? And of claim A?
2. Storm or hail: should the cause change the payout?
3. How much should **tone**, **writing** and **language** take away, if the claim desk works well?

## Step 6 · SHAP: what did each input add or take away?

**What this block does:**
- It builds the **reference claim** (you can change it with the menus).
- SHAP then tries every mix of the two claims: for each input it takes the value of claim A or of the reference. It sends each mix through the claim desk.
- It splits the difference in payout fairly over the inputs (Shapley values, from game theory).

**What you will see:** the SHAP value per input, every mix that SHAP tried with its payout, and a waterfall chart. At the bottom of the chart, the payout of the reference claim; at the top, the payout of claim A. Each bar is one input: red adds money, blue takes it away. Inputs that are the same in both claims get 0.

**If a number looks odd:** look at the list of mixes. The model sometimes gives a different answer by chance, for example NEED MORE INFORMATION. To ask the model again, run Step 3 again (it empties the list of results), then Steps 4 and 6.

**Cost:** one run per mix. With 4 inputs that differ: 16 runs, about 48 calls. With all 6 different: 64 runs. The list of results means no mix is sent twice.

In [ ]:
# @title Step 6 · SHAP: explain claim A against the reference claim
reference_cause = "hail"        # @param ["hail", "storm", "burst pipe", "burglary", "old roof"]
reference_tone = "polite"       # @param ["polite", "angry"]
reference_writing = "neat"      # @param ["neat", "sloppy"]
reference_language = "Dutch"    # @param ["English", "Dutch"]

import numpy as np
import shap

REFERENCE = {**CLAIM_A, "cause": reference_cause, "tone": reference_tone, "writing": reference_writing, "language": reference_language}

def as_codes(claim):
    """SHAP works with numbers: each value becomes its place in the list of possible values."""
    return np.array([[INPUTS[name].index(str(claim[name])) for name in NAMES]])

def payout_for(rows):
    """For SHAP: rows of codes in, payouts out. Each row is one claim, sent through the claim desk."""
    return np.array([desk_result({name: INPUTS[name][int(code)] for name, code in zip(NAMES, row)})["payout"]
                     for row in rows], dtype=float)

calls_before = len(calls)
explainer = shap.explainers.Exact(payout_for, shap.maskers.Independent(as_codes(REFERENCE), max_samples=1))
explanation = explainer(as_codes(CLAIM_A))
explanation.feature_names = NAMES
explanation.display_data = np.array([[str(CLAIM_A[name]) for name in NAMES]], dtype=object)

print("Reference claim:", REFERENCE)
print("Claim A:        ", CLAIM_A)
print(f"Payout of the reference claim: EUR {desk_result(REFERENCE)['payout']:,}")
print(f"Payout of claim A:             EUR {desk_result(CLAIM_A)['payout']:,}\n")
for name, value in zip(NAMES, explanation.values[0]):
    print(f"  {name:8} {str(REFERENCE[name]):>9} → {str(CLAIM_A[name]):9}  SHAP: EUR {value:+,.0f}")
print(f"\nThis step used {len(calls) - calls_before} calls. Total so far: {len(calls)} calls, ${total_cost():.4f}.")
print("\nWhat SHAP tried: every mix of claim A and the reference claim (only the inputs that differ)")
differ = [name for name in NAMES if str(CLAIM_A[name]) != str(REFERENCE[name])]
print("  " + "  ".join(f"{name:>8}" for name in differ))
for key, result in results.items():
    mix = dict(zip(NAMES, key))
    if all(mix[name] in (str(CLAIM_A[name]), str(REFERENCE[name])) for name in differ) and        all(mix[name] == str(CLAIM_A[name]) for name in NAMES if name not in differ):
        print("  " + "  ".join(f"{mix[name]:>8}" for name in differ) + f"   →  {result['decision']:<22} EUR {result['payout']:,}")
shap.plots.waterfall(explanation[0])

## Step 7 · Predict first: on paper, laptops closed

**Claim B** is about an **old roof**: the tiles are worn out after many years. The policy does not cover wear and tear, so the claim desk should not pay (NOT COVERED, or NEED MORE INFORMATION).

DiCE will look for the **smallest change** in the email that gets claim B a payout.

**Write down, before you run Step 8:**
1. Which change would get claim B covered?
2. Could a customer make that change without lying? What does that mean for the claim desk?

## Step 8 · DiCE: what is the smallest change that gets claim B covered?

**What this block does:**
- It sends claim B through the claim desk.
- DiCE tries random changes to claim B: first one input at a time, then two, and so on. It keeps the changes that turn the decision into COVERED, and shows up to three of them.
- All six inputs may change: cause, damage, cost, tone, writing and language.
- You can choose another claim B with the menus on the right of the block.

**What you will see:** claim B, then a table with up to three changed claims. A dash (-) means: unchanged.

**Cost:** DiCE needs about 10 to 30 runs, so 30 to 90 calls; a few cents at most. The list of results means no claim is sent twice.

In [ ]:
# @title Step 8 · DiCE: find the smallest change that gets claim B covered
claim_b_cause = "old roof"   # @param ["hail", "storm", "burst pipe", "burglary", "old roof"]
claim_b_damage = "roof"      # @param ["roof", "belongings"]
claim_b_cost = "2500"        # @param ["800", "2500", "9000", "30000"]
claim_b_tone = "polite"      # @param ["polite", "angry"]
claim_b_writing = "neat"     # @param ["neat", "sloppy"]
claim_b_language = "English" # @param ["English", "Dutch"]

import pandas as pd
import dice_ml

CLAIM_B = {"cause": claim_b_cause, "damage": claim_b_damage, "cost": claim_b_cost,
           "tone": claim_b_tone, "writing": claim_b_writing, "language": claim_b_language}
first = desk_result(CLAIM_B)
show_box("The email of claim B", write_email(CLAIM_B))
print(f"Claim B: {first['decision']}, payout EUR {first['payout']:,}. {first['reason']}\n")

class ClaimDesk:
    """For DiCE: a table of claims in, the chance of COVERED out (here simply 0 or 1)."""
    def predict_proba(self, table):
        covered = [1.0 if desk_result(dict(zip(NAMES, map(str, row))))["covered"] else 0.0
                   for row in table[NAMES].itertuples(index=False)]
        return np.array([[1 - c, c] for c in covered])
    def predict(self, table):
        return (self.predict_proba(table)[:, 1] >= 0.5).astype(int)

calls_before = len(calls)
data = dice_ml.Data(features={name: INPUTS[name] for name in NAMES}, continuous_features=[], outcome_name="covered")
dice = dice_ml.Dice(data, dice_ml.Model(model=ClaimDesk(), backend="sklearn", model_type="classifier"), method="random")
found = dice.generate_counterfactuals(pd.DataFrame([CLAIM_B]), total_CFs=3, desired_class=1, sample_size=10,
                                      features_to_vary=NAMES, random_seed=1)
found.visualize_as_dataframe(show_only_changes=True)
print(f"This step used {len(calls) - calls_before} calls. Total so far: {len(calls)} calls, ${total_cost():.4f}.")

## Step 9 · Compare and discuss

Put your results next to those of another pair.

1. **SHAP:** did tone, writing or language take money away? If so, how much? Did you get the same numbers as the other pair?
2. **DiCE:** which change got claim B covered? Could a customer make it with a lie, for example by writing *storm* instead of *old roof*? Where should a person or an expert step in?
3. **The randomness:** were the three runs in Step 4 the same? What does that mean for testing an AI system once?
4. **For a firm:** would you let this claim desk decide without a person? Under the AI Act, would it be high risk? (Look back at the lecture.)

## Try this, each pair something different

- Choose another claim A in Step 4 (for example burglary, or a higher cost) and run Steps 4 and 6 again.
- Add a sentence to the angry closing, for example *"My lawyer is reading along."* Does the payout change?
- Look inside: open Step 2 (*Show code*) and read the instructions and the policy document. Can you explain what you found?
- Change agent 2's instructions in Step 2 so that it ignores the tone. Run the test again.
- Choose another claim B in Step 8, for example a claim the desk should pay but does not, and see what DiCE changes.

In [ ]:
# @title The cost of this notebook so far
print(f"Calls to the model: {len(calls)}")
print(f"Cost: ${total_cost():.4f} (budget: ${MAX_COST_TOTAL_USD})")
print(f"Different claims sent to the claim desk: {len(results)}")